In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# Q4.1: PHÁT HIỆN CÁC NGÀY CÓ DẤU HIỆU QUÁ NHIỆT
q4_1_overheat = spark.sql('''
    SELECT
        date,
        COUNT(*) AS tong_ban_ghi,
        SUM(
            CASE
                WHEN Oil_temperature >= 75
                THEN 1 ELSE 0
            END) AS so_lan_vuot_75C,
        SUM(
            CASE
                WHEN Oil_temperature >= 80
                THEN 1 ELSE 0
            END) AS so_lan_vuot_80C,
        ROUND(MAX(Oil_temperature), 2) AS nhiet_do_cao_nhat,
        ROUND(AVG(Oil_temperature), 2) AS nhiet_do_trung_binh
    FROM sensor
    WHERE COMP = 0
    GROUP BY date
    HAVING so_lan_vuot_75C > 0
    ORDER BY so_lan_vuot_75C DESC
    LIMIT 10''')
print("\n========== EXECUTION PLAN ==========")
q4_1_overheat.explain(True)
print("\n--- TOP 10 NGÀY CÓ DẤU HIỆU QUÁ NHIỆT ---")
df_q4_1 = q4_1_overheat.toPandas()
try: display(df_q4_1)
except NameError: print(df_q4_1.to_string(index=False))